# Part 2: Streaming application using Spark Structured Streaming  
In this task, you will implement Spark Structured Streaming to consume the data from task 1 and perform a prediction.    
Important:   
-	This task uses PySpark Structured Streaming with PySpark Dataframe APIs and PySpark ML.  
-	You also need your pipeline model from A2A to make predictions and persist the results.  

1.	Write code to create a SparkSession, which 1) uses four cores with a proper application name; 2) use the Melbourne timezone; 3) ensure a checkpoint location has been set.


In [1]:
# Import libraries needed from pyspark
from pyspark import SparkConf
from pyspark import SparkContext # Spark
from pyspark.sql import SparkSession # Spark SQL

#Kafka requirements
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.5.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 pyspark-shell'

# Create Spark Configuration Object
master = "local[4]" # 4 Cores as per the spec
app_name = "35090197_Assignment_2B_Streaming"
checkpoint_dir = "./checkpoints"
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

# Create SparkSession
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
spark.conf.set('spark.sql.session.timeZone', "Australia/Melbourne") # Setting the timezone to Melbourne time.
spark.conf.set("spark.sql.streaming.checkpointLocation", checkpoint_dir) # Setting the checkpoint location for streaming data.
sc = spark.sparkContext
sc.setLogLevel('ERROR')

2.	Write code to define the data schema for the data files, following the data types suggested in the metadata file. Load the static datasets (e.g. building information) into data frames. (You can reuse your code from 2A.)


In [2]:
from pyspark.sql.types import StructType,StructField
from pyspark.sql.types import IntegerType, StringType, DecimalType, TimestampType

# 1. Meters Table Schema
meters_schema = StructType([
    StructField("building_id", IntegerType(), True),
    StructField("meter_type", StringType(), True),
    StructField("ts", TimestampType(), True),
    StructField("value", DecimalType(10,4), True),
    StructField("row_id", IntegerType(), True)
])

# 2. Buildings Table Schema
buildings_info_schema = StructType([
    StructField("site_id", IntegerType(), True),
    StructField("building_id", IntegerType(), True),
    StructField("primary_use", StringType(), True),
    StructField("square_feet", IntegerType(), True),
    StructField("floor_count", IntegerType(), True),
    StructField("row_id", IntegerType(), True),
    StructField("year_built", IntegerType(), True),
    StructField("latent_y", DecimalType(10,2), True),
    StructField("latent_s", DecimalType(11,8), True),
    StructField("latent_r", DecimalType(10,2), True)
])


meters_df = spark.read.csv("new_meters.csv", header=True, schema=meters_schema)
buildings_df = spark.read.csv("new_building_information.csv", header=True, schema=buildings_info_schema)

# 3. Weather Table Schema
# Since custom schema definition is not able to handle cloud_coverage and wind_direction values appropriately,
# we will first cast it as StringType and then explicitly cast it as required.
import pyspark.sql.functions as F

# Weather Table Schema
weather_schema_static = StructType([
    StructField("site_id", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("air_temperature", DecimalType(5,2), True),
    StructField("cloud_coverage", StringType(), True),
    StructField("dew_temperature", DecimalType(5,2), True),
    StructField("sea_level_pressure", DecimalType(7,2), True),
    StructField("wind_direction", StringType(), True),
    StructField("wind_speed", DecimalType(5,2), True)
])

weather_df = spark.read.csv("weather.csv", header=True, schema=weather_schema_static)
weather_df = weather_df \
    .withColumn("cloud_coverage", F.when(F.col("cloud_coverage") == "", None).otherwise(F.col("cloud_coverage").cast("int"))) \
    .withColumn("wind_direction", F.when(F.col("wind_direction") == "", None).otherwise(F.col("wind_direction").cast("int")))

# Training the imputer model for using later on the streaming data:
# Define the Imputer -- and setting "mean" as the strategy.
from pyspark.ml.feature import Imputer
inputCols_weather=["air_temperature", "cloud_coverage", "dew_temperature", "sea_level_pressure", "wind_direction", "wind_speed"]
imputer = Imputer(
    inputCols=inputCols_weather,
    outputCols=["air_temperature_imputed", "cloud_coverage_imputed", "dew_temperature_imputed", "sea_level_pressure_imputed",
                "wind_direction_imputed", "wind_speed_imputed"]
).setStrategy("mean")
# Fit model to dataframe/dataset
imputer_model = imputer.fit(weather_df)

# Calculation of peak_offpeak months (from A2A) -- from static data to be used with streaming data.
# First, we generate the monthly averages
weather_df = imputer_model.transform(weather_df)
monthly_avg = (
    weather_df.withColumn("month", F.month(F.col("timestamp")))
    .groupBy("month")
    .agg(F.avg("air_temperature_imputed").alias("avg_air_temp"))
    .orderBy("avg_air_temp")
)
# Then we get the top 3 hottest and coldest months
top3_hottest = monthly_avg.orderBy(F.col("avg_air_temp").desc()).limit(3)
top3_coldest = monthly_avg.orderBy(F.col("avg_air_temp").asc()).limit(3)
# Now we label and combine these values
top3_hottest = top3_hottest.withColumn("peak_offpeak", F.lit("peak"))
top3_coldest= top3_coldest.withColumn("peak_offpeak", F.lit("peak"))
peak_df = top3_hottest.union(top3_coldest)
peak_df = peak_df.drop("avg_air_temp")

3.	Using the Kafka topic from the producer in Task 1, ingest the streaming data into Spark Streaming, assuming all data comes in the String format. Except for the 'weather_ts' column, you shall receive it as an Int type. Load the new building information CSV file into a dataframe. Then, the data frames should be transformed into the proper formats following the metadata file schema, similar to assignment 2A.


In [3]:
# [NOTE]: Please note that the new_building and new_meters dfs have been loaded already in the above code block.

In [4]:
from pyspark.sql.functions import explode
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.types import *

topic = "weather5s"
hostip = "kafka"

df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", f"{hostip}:9092") \
    .option("subscribe", topic) \
    .load()

df = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")

# Converting the incoming stream into dataframe based on the schema.
# Note that, were are receiving a list from the producer, so the schema and other transformations are done accordingly
# Since custom schema definition is not able to handle site_id, cloud_coverage and wind_direction values appropriately,
# we will first cast it as StringType and then explicitly cast it as required.
weather_schema_stream = ArrayType(StructType([
    StructField("site_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("air_temperature", DecimalType(5,2), True),
    StructField("cloud_coverage", StringType(), True),
    StructField("dew_temperature", DecimalType(5,2), True),
    StructField("sea_level_pressure", DecimalType(7,2), True),
    StructField("wind_direction", StringType(), True),
    StructField("wind_speed", DecimalType(5,2), True),
    StructField('weather_ts', IntegerType(), True)
]))

df = df.select(F.from_json(F.col("value").cast("string"), weather_schema_stream).alias('parsed_value'))
df = df.select(F.explode(F.col("parsed_value")).alias('unnested_value'))
df = df.select(
    F.col("unnested_value.site_id").cast("integer").alias("site_id"),
    F.col("unnested_value.timestamp").alias("timestamp"),
    F.col("unnested_value.air_temperature").alias("air_temperature"),
    F.col("unnested_value.cloud_coverage").cast("integer").alias("cloud_coverage"),
    F.col("unnested_value.dew_temperature").alias("dew_temperature"),
    F.col("unnested_value.sea_level_pressure").alias("sea_level_pressure"),
    F.col("unnested_value.wind_direction").cast("integer").alias("wind_direction"),
    F.col("unnested_value.wind_speed").alias("wind_speed"),
    F.col("unnested_value.weather_ts").alias("weather_ts")
)
df.printSchema() #verification of approciate schema

root
 |-- site_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- air_temperature: decimal(5,2) (nullable = true)
 |-- cloud_coverage: integer (nullable = true)
 |-- dew_temperature: decimal(5,2) (nullable = true)
 |-- sea_level_pressure: decimal(7,2) (nullable = true)
 |-- wind_direction: integer (nullable = true)
 |-- wind_speed: decimal(5,2) (nullable = true)
 |-- weather_ts: integer (nullable = true)



4.	Use a watermark on weather_ts, if data points are received 5 seconds late, discard the data.

In [5]:
# withWatermark() only accepts TimestampType and so, we cast it from Integer to Timestamp as necessary.

# we use fromm_unixtime to let spark know it is UNIX seconds from UTC start time.
# We dont necessarily have to do this but it is safer.
df = df.withColumn("weather_ts", F.from_unixtime(F.col("weather_ts")).cast("timestamp"))

df = df.withWatermark("weather_ts", "5 seconds")

5.	Perform the necessary transformation you used in A2A. (note: every student may have used different features, feel free to reuse the code you have written in A2A. If you built an end-to-end pipeline, you can ignore this task.) 

In [6]:
# Meters data & Buildings data transformations
meters_df = meters_df.withColumn("hour", F.hour(F.col("ts")))

# We have 4 intervals so we will use the values 0,1,2 & 3 for representation.
meters_df = meters_df.withColumn(
    "interval",
    F.when((F.col("hour") >= 0) & (F.col("hour") < 6), 0)
    .when((F.col("hour") >= 6) & (F.col("hour") < 12), 1)
    .when((F.col("hour") >= 12) & (F.col("hour") < 18), 2)
    .otherwise(3)
)
meters_df = meters_df.drop("hour")
meters_df = meters_df.groupBy("building_id", "interval") \
    .agg(F.sum("value").alias("total_energy")).orderBy("building_id", "interval")

buildings_df = buildings_df.drop("row_id") 
meters_and_buildings = meters_df.join(buildings_df, on="building_id", how="left")
# remove useless rows -- not needed for feature_df.
meters_and_buildings = meters_and_buildings.drop("latent_y", "latent_r", "month")
meters_and_buildings.show(3)
# just some quick clean-up of memory
import gc
gc.collect()

+-----------+--------+------------+-------+-----------+-----------+-----------+----------+----------+
|building_id|interval|total_energy|site_id|primary_use|square_feet|floor_count|year_built|  latent_s|
+-----------+--------+------------+-------+-----------+-----------+-----------+----------+----------+
|        914|       0|2880576.8121|      9|  Education|     229973|          1|      1998|5.36167670|
|        959|       0|2077020.8000|      9|  Education|     128536|          1|      2006|5.10902500|
|       1350|       2|2007076.4704|     15|  Education|     149762|          1|      2011|5.17540170|
+-----------+--------+------------+-------+-----------+-----------+-----------+----------+----------+
only showing top 3 rows



138

In [7]:
# Weather Data transformations to feature_df

# imputing missing values using the pre-trained imputer.
df = imputer_model.transform(df)
# removing the columns with null values and renaming imputed columns.
for col_name in inputCols_weather:
    df = df.drop(col_name).withColumnRenamed(f"{col_name}_imputed", col_name)
    
# Adding peak_offpeak
# Add month values to the df (using peak_df value from above)
df = df.withColumn("month", F.month("timestamp"))
df = df.join(F.broadcast(peak_df), on="month", how='left')
# Fill nulls with "off-peak"
df = df.withColumn(
    "peak_offpeak",
    F.coalesce(F.col("peak_offpeak"), F.lit("off-peak"))
)
df = df.drop("month")

# we need to add interval to df to be able to accurately join with meters_and_buildings.
df = df.withColumn("hour", F.hour(F.col("timestamp")))
df = df.withColumn(
    "interval",
    F.when((F.col("hour") >= 0) & (F.col("hour") < 6), 0)
    .when((F.col("hour") >= 6) & (F.col("hour") < 12), 1)
    .when((F.col("hour") >= 12) & (F.col("hour") < 18), 2)
    .otherwise(3)
)
df = df.drop("hour")

df = df.join(meters_and_buildings, on=["site_id","interval"], how="left")
df = df.withColumn("is_weekend", F.dayofweek("timestamp").isin([7, 1]).cast("int")) #0 or 1 values.
df = df.withColumn("air_dew_diff", (F.col("air_temperature") - F.col("dew_temperature")))
df = df.withColumn("date", F.to_date(F.col("timestamp")))
df = df.drop("timestamp", "air_temperature", "dew_temperature")

In [8]:
df.printSchema() # schema verification

root
 |-- site_id: integer (nullable = true)
 |-- interval: integer (nullable = false)
 |-- weather_ts: timestamp (nullable = true)
 |-- cloud_coverage: integer (nullable = true)
 |-- sea_level_pressure: decimal(7,2) (nullable = true)
 |-- wind_direction: integer (nullable = true)
 |-- wind_speed: decimal(5,2) (nullable = true)
 |-- peak_offpeak: string (nullable = false)
 |-- building_id: integer (nullable = true)
 |-- total_energy: decimal(20,4) (nullable = true)
 |-- primary_use: string (nullable = true)
 |-- square_feet: integer (nullable = true)
 |-- floor_count: integer (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- latent_s: decimal(11,8) (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- air_dew_diff: decimal(6,2) (nullable = true)
 |-- date: date (nullable = true)



6.	Load your pipeline model and perform the following aggregations:  
a)	Print the prediction from your model as a stream comes in.  
b)	Every 7 seconds, print the total energy consumption for each 6-hour interval, aggregated by building, and print 20 records. (Note: This is simulating energy data each day in a week)  
c)	Every 14 seconds, for each site, print the daily total energy consumption.  

In [11]:
#Loading the Pipeline Model From the filesystem
from pyspark.ml import PipelineModel
pipelineModel = PipelineModel.load('A2A_prediction_model_RF')
print(pipelineModel.stages[-1]._java_obj.paramMap()) #just a verification print.

gc.collect() # just some quick memory clean up.

{
	RandomForestRegressor_e9f1b967bcc4-featuresCol: features,
	RandomForestRegressor_e9f1b967bcc4-labelCol: total_energy,
	RandomForestRegressor_e9f1b967bcc4-maxBins: 20,
	RandomForestRegressor_e9f1b967bcc4-maxDepth: 11,
	RandomForestRegressor_e9f1b967bcc4-numTrees: 10,
	RandomForestRegressor_e9f1b967bcc4-seed: 2025
}


352

In [12]:
# Predictions from stream data
stream_predictions =  pipelineModel.transform(df).drop("features")
stream_predictions.printSchema()

root
 |-- site_id: integer (nullable = true)
 |-- interval: integer (nullable = false)
 |-- weather_ts: timestamp (nullable = true)
 |-- cloud_coverage: integer (nullable = true)
 |-- sea_level_pressure: decimal(7,2) (nullable = true)
 |-- wind_direction: integer (nullable = true)
 |-- wind_speed: decimal(5,2) (nullable = true)
 |-- peak_offpeak: string (nullable = false)
 |-- building_id: integer (nullable = true)
 |-- total_energy: decimal(20,4) (nullable = true)
 |-- primary_use: string (nullable = true)
 |-- square_feet: integer (nullable = true)
 |-- floor_count: integer (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- latent_s: decimal(11,8) (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- air_dew_diff: decimal(6,2) (nullable = true)
 |-- date: date (nullable = true)
 |-- peak_offpeak_index: double (nullable = false)
 |-- primary_use_index: double (nullable = false)
 |-- peak_offpeak_vec: vector (nullable = true)
 |-- primary_use_vec: vector (nul

In [13]:
# Calculation for 6B
# Aggregate predictions per building and interval
interval_agg_df = stream_predictions \
    .groupBy(window("weather_ts", "7 seconds"), "building_id", "interval") \
    .agg(F.sum("prediction").alias("6hr_building_NRGy_total"))

# Calculation for 6C
# Aggregate predictions per site and date
site_agg_df = stream_predictions \
    .groupBy(window("weather_ts", "14 seconds"), "site_id", "date") \
    .agg(F.sum("prediction").alias("daily_site_NRGy_total"))

In [14]:
# 6a Print the prediction from your model as a stream comes in.
query_6A = stream_predictions.writeStream \
    .outputMode("append") \
    .option("truncate", "false") \
    .format("console") \
    .start()

gc.collect()

105

In [15]:
# 6b
query_6B = interval_agg_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .option("numRows", 20) \
    .trigger(processingTime="7 seconds") \
    .start()

gc.collect()

49

In [16]:
# 6c
query_6C = site_agg_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime="14 seconds") \
    .start()

gc.collect()

42

7.	Save the data from 6 to Parquet files as streams. (Hint: Parquet files support streaming writing/reading. The file keeps updating while new batches arrive.)

In [17]:
# 7a(save 6a)
query_file_sink_7a = stream_predictions.writeStream.format("parquet")\
    .outputMode("append")\
    .option("path", "parquet/df_6a")\
    .option("checkpointLocation", "parquet/df_6a/checkpoint") \
    .start()

In [18]:
# 7b(save 6b)
query_file_sink_7b = (
    interval_agg_df.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", "parquet/df_6b")
    .option("checkpointLocation", "parquet/df_6b/checkpoint")
    .start()
)

In [19]:
# 7c(save 6c)
query_file_sink_7c = (
    site_agg_df.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", "parquet/df_6c")
    .option("checkpointLocation", "parquet/df_6c/checkpoint")
    .start()
)

8.	Read the parquet files from task 7 as data streams and send them to Kafka topics with appropriate names.
(Note: You shall read the parquet files as a streaming data frame and send messages to the Kafka topic when new data appears in the parquet file.)

In [20]:
# give spark enough time to generate parquet files in the file system.
import time
time.sleep(120)

In [21]:
# Stream 1
schema_8a = stream_predictions.schema
datastream_8a = spark\
                .readStream\
                .schema(schema_8a)\
                .format("parquet")\
                .load("parquet/df_6a")

query_8a = datastream_8a \
  .writeStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", f'{hostip}:9092') \
  .option("topic", "8a") \
  .start()

In [22]:
# Stream 2
from pyspark.sql.functions import to_json, struct

schema_8b = interval_agg_df.schema
datastream_8b = spark\
                .readStream\
                .schema(schema_8b)\
                .format("parquet")\
                .load("parquet/df_6b")

datastream_8b_kafka = datastream_8b.selectExpr("CAST(building_id AS STRING) AS key", "to_json(struct(*)) AS value")

query_8b = datastream_8b_kafka \
  .writeStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", f'{hostip}:9092') \
  .option("topic", "8b") \
  .start()

In [23]:
# Stream 3
schema_8c = site_agg_df.schema
datastream_8c = spark\
                .readStream\
                .schema(schema_8c)\
                .format("parquet")\
                .load("parquet/df_6c")

datastream_8c_kafka = datastream_8c.selectExpr("CAST(site_id AS STRING) AS key", "to_json(struct(*)) AS value")

query_8c = datastream_8c_kafka \
  .writeStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", f'{hostip}:9092') \
  .option("topic", "8c") \
  .start()

Code block to stop all queries.

In [24]:
# query_6A.stop() 
# query_6B.stop()
# query_6C.stop()
# query_file_sink_7a.stop()
# query_file_sink_7b.stop()
# query_file_sink_7c.stop()
# query_8a.stop() 
# query_8b.stop()
# query_8c.stop()

gc.collect()

324

### References

<!-- Apache Spark. ( Accessed 2025 ). <i>pyspark.SparkContext.setCheckpointDir</i>. Apache Spark API Reference. https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.SparkContext.setCheckpointDir.html -->

ApacheSpark. ( Accessed 2025 ). <i>spark.sql.streaming.checkpointLocation</i>. Spark Configuration.
https://spark.apache.org/docs/latest/configuration.html#memory-management:~:text=spark.sql.streaming.checkpointLocation

ApacheSpark. ( Accessed 2025 ). <i>Structured Streaming Programming Guide</i>. spark.apache.org.<br>
https://spark.apache.org/docs/3.5.1/structured-streaming-programming-guide.html

ApacheSpark. ( Accessed 2025 ). <i>Structured Streaming + Kafka Integration Guide (Kafka broker version 0.10.0 or higher)</i>spark.apache.org.<br>
https://spark.apache.org/docs/3.5.1/structured-streaming-kafka-integration.html

Bhandani, N. ( 2021 ). <i>Apache Spark Structured Streaming — Output Sinks (3 of 6)</i>. Medium.com<br>
https://medium.com/expedia-group-tech/apache-spark-structured-streaming-output-sinks-3-of-6-ed3247545fbc

Bhandani, N. ( 2021 ). <i>Apache Spark Structured Streaming — Input Sources (2 of 6)</i>. Medium.com<br>
https://medium.com/expedia-group-tech/apache-spark-structured-streaming-input-sources-2-of-6-6a72f798838c

Das, T. ( 2017 ). <i>Event-time Aggregation and Watermarking in Apache Spark’s Structured Streaming</i> Databricks<br>
https://www.databricks.com/blog/2017/05/08/event-time-aggregation-watermarking-apache-sparks-structured-streaming.html

